<a href="https://colab.research.google.com/github/Cristian20052611/Inteligencia-Artificisl-II/blob/main/prediccion_precios_casasl.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. Importar las librerías

Antes de hacer cualquier cosa necesito las herramientas. Aquí dejo anotado
para qué me sirve cada una, porque al principio a uno se le olvida por qué
se importa cada librería.

In [ ]:
import pandas as pd  # la uso para cargar el csv y manejar los datos como una tabla (DataFrame), es mi herramienta principal para todo lo tabular
import numpy as np  # me sirve para operaciones numéricas rápidas (arreglos, raíz cuadrada para el RMSE, etc)
import matplotlib.pyplot as plt  # con esta hago las gráficas básicas (dispersión, barras)
import seaborn as sns  # esta es como un "maquillaje" de matplotlib, la uso para el mapa de calor de correlaciones porque se ve más claro

from sklearn.model_selection import train_test_split  # función que parte mis datos en un conjunto de entrenamiento y uno de prueba, así no hago trampa evaluando con los mismos datos que entrené
from sklearn.linear_model import LinearRegression  # el primer modelo que voy a comparar: una línea recta que intenta ajustar el precio
from sklearn.tree import DecisionTreeRegressor  # el segundo modelo: un árbol que va haciendo preguntas tipo "¿el área es mayor a X?" hasta llegar a una predicción
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score  # estas tres métricas son las que voy a usar para decir "qué tan bueno" es cada modelo

# esta línea es solo estética: hace que las gráficas de seaborn se vean más limpias por defecto
sns.set_style("whitegrid")


## 2. Cargar el dataset

Voy a leer el archivo `Housing.csv` que descargué de Kaggle. En Colab lo subo
manualmente (ícono de carpeta a la izquierda -> Upload) o lo monto desde mi
Google Drive. Dejo las dos opciones comentadas por si acaso.

In [ ]:
# Opción A: si ya subí el archivo directo a la sesión de Colab (lo más simple para este ejercicio)
ruta_archivo = "Housing.csv"  # guardo la ruta en una variable para no repetir el string por todo el notebook

# Opción B (la dejo comentada): si prefiero montar mi Google Drive para no subir el archivo cada vez que se reinicia la sesión
# from google.colab import drive          # esta librería solo existe dentro de Colab, por eso la importo solo si la voy a usar
# drive.mount('/content/drive')           # esto me pide permiso y conecta mi Drive a la carpeta /content/drive
# ruta_archivo = "/content/drive/MyDrive/datasets/Housing.csv"  # ajusto la ruta a donde tenga guardado el csv en mi Drive

datos = pd.read_csv(ruta_archivo)  # aquí sí cargo el csv como un DataFrame de pandas, esto es literalmente "abrir la hoja de cálculo" en Python
datos.head()  # imprimo las primeras 5 filas solo para verificar visualmente que el archivo cargó bien y las columnas tienen sentido


## 3. Explorar los datos (EDA)

Antes de entrenar cualquier modelo, necesito entender qué tengo entre manos:
cuántas filas hay, si hay datos vacíos, cómo se distribuye el precio, y qué
tan relacionadas están las variables numéricas entre sí.

In [ ]:
print("Filas y columnas:", datos.shape)  # shape me da (filas, columnas), así sé de una vez el tamaño del dataset (545 filas, 13 columnas según Kaggle)
datos.info()  # esto me muestra el tipo de dato de cada columna (número o texto) y si hay nulos, es mi "chequeo de salud" inicial del dataset


In [ ]:
datos.isnull().sum()  # sumo los valores nulos por columna; si todo da 0 significa que no tengo que preocuparme por imputar datos faltantes


In [ ]:
datos.describe()  # me da media, min, max, desviación estándar de las columnas numéricas, útil para detectar valores raros (ej. una casa de área 1 o de precio 0)


In [ ]:
plt.figure(figsize=(6, 4))  # abro una figura nueva y le doy un tamaño razonable para que no se vea aplastada
sns.histplot(datos["price"], kde=True)  # dibujo el histograma del precio con una curva de densidad encima, quiero ver si los precios se concentran en un rango o si hay casas carísimas que se salen del montón
plt.title("Distribución del precio de las casas")  # le pongo título para no perderme cuando vuelva a ver esta gráfica en un mes
plt.xlabel("Precio")  # aclaro qué es el eje X
plt.show()  # esta línea es la que realmente "pinta" la gráfica en pantalla


In [ ]:
columnas_numericas = ["price", "area", "bedrooms", "bathrooms", "stories", "parking"]  # separo solo las columnas que ya son números, porque a la correlación no le puedo meter texto todavía
matriz_correlacion = datos[columnas_numericas].corr()  # calculo qué tan relacionada está cada variable numérica con las demás (y en particular con el precio)

plt.figure(figsize=(7, 5))  # nueva figura para el mapa de calor
sns.heatmap(matriz_correlacion, annot=True, cmap="coolwarm", fmt=".2f")  # annot=True me muestra el número dentro de cada celda, así no tengo que adivinar el color
plt.title("Correlación entre variables numéricas y el precio")  # título para recordar qué estoy mirando
plt.show()  # muestro el mapa de calor


**Mi lectura de esta parte:** el área es la variable numérica que más se
relaciona con el precio (tiene la correlación más alta), lo cual tiene sentido
porque una casa más grande normalmente cuesta más. Bathrooms y stories también
aportan, pero menos. Esto ya me da una pista de qué variables van a "pesar" más
dentro de los modelos.

## 4. Preprocesamiento de datos

El dataset tiene columnas categóricas tipo texto (`yes`/`no`) y una columna
con varias categorías (`furnishingstatus`). Los modelos de scikit-learn solo
entienden números, así que tengo que convertir todo eso antes de entrenar.

In [ ]:
columnas_si_no = ["mainroad", "guestroom", "basement", "hotwaterheating", "airconditioning", "prefarea"]  # estas son todas las columnas que solo tienen "yes" o "no" como valor

for columna in columnas_si_no:  # recorro cada una de esas columnas una por una
    datos[columna] = datos[columna].map({"yes": 1, "no": 0})  # reemplazo "yes" por 1 y "no" por 0, así el modelo lo puede leer como número (1 = tiene la característica, 0 = no la tiene)

datos[columnas_si_no].head()  # reviso las primeras filas para confirmar que el reemplazo quedó bien y no me quedó ningún texto suelto


In [ ]:
# furnishingstatus tiene tres valores: "furnished", "semi-furnished", "unfurnished"
# los ordeno de menos a más amoblado porque sí tiene un orden lógico (no es una categoría sin orden como un color)
orden_amoblado = {"unfurnished": 0, "semi-furnished": 1, "furnished": 2}
datos["furnishingstatus"] = datos["furnishingstatus"].map(orden_amoblado)  # convierto el texto a su número correspondiente según el diccionario de arriba

datos["furnishingstatus"].value_counts()  # cuento cuántas casas quedaron en cada categoría, para confirmar que el mapeo no me dejó ningún NaN por un valor mal escrito


In [ ]:
caracteristicas = datos.drop(columns=["price"])  # esta es mi matriz X: todas las columnas MENOS el precio, porque el precio es lo que quiero predecir, no algo que el modelo deba usar como pista
objetivo = datos["price"]  # esta es mi variable y: lo que el modelo tiene que aprender a adivinar

caracteristicas.head()  # miro cómo quedaron mis variables de entrada, ya todas numéricas


## 5. Dividir en entrenamiento y prueba

Divido los datos en dos partes: una para que el modelo "estudie" (train) y
otra que el modelo nunca ve durante el entrenamiento (test), para simular
cómo se comportaría con casas nuevas.

In [ ]:
X_entrenamiento, X_prueba, y_entrenamiento, y_prueba = train_test_split(
    caracteristicas,       # mis variables de entrada
    objetivo,              # mi variable a predecir (el precio)
    test_size=0.2,         # dejo el 20% de las casas solo para evaluar, el 80% restante es para entrenar
    random_state=42        # fijo una "semilla" para que la división sea siempre la misma cada vez que corro el notebook (así puedo comparar resultados de forma justa)
)

print("Casas para entrenar:", X_entrenamiento.shape[0])  # imprimo cuántas filas quedaron para entrenar
print("Casas para probar:", X_prueba.shape[0])  # imprimo cuántas filas quedaron para probar, solo para confirmar que el 80/20 se aplicó bien


## 6. Modelo 1 — Regresión Lineal

Este modelo asume que el precio se puede explicar como una combinación lineal
(una suma con pesos) de las variables de entrada. Es el modelo más simple e
interpretable, por eso empiezo por acá.

In [ ]:
modelo_lineal = LinearRegression()  # creo el modelo vacío, todavía no ha "visto" ningún dato
modelo_lineal.fit(X_entrenamiento, y_entrenamiento)  # aquí es donde realmente aprende: calcula los pesos que mejor ajustan el precio en los datos de entrenamiento

predicciones_lineal = modelo_lineal.predict(X_prueba)  # le pido que prediga el precio de las casas de prueba, que el modelo nunca vio
predicciones_lineal[:5]  # muestro las primeras 5 predicciones solo para verlas "crudas" antes de compararlas con el precio real


In [ ]:
mae_lineal = mean_absolute_error(y_prueba, predicciones_lineal)  # promedio del error absoluto: en promedio, ¿cuánto le erra el modelo en pesos/dólares?
rmse_lineal = np.sqrt(mean_squared_error(y_prueba, predicciones_lineal))  # parecido al MAE pero castiga más fuerte los errores grandes, por eso le saco raíz al final para que quede en la misma unidad que el precio
r2_lineal = r2_score(y_prueba, predicciones_lineal)  # me dice qué porcentaje de la variación del precio logra "explicar" el modelo (1.0 sería perfecto, 0 sería que no explica nada)

print(f"Regresión Lineal -> MAE: {mae_lineal:,.0f} | RMSE: {rmse_lineal:,.0f} | R2: {r2_lineal:.3f}")  # imprimo las tres métricas juntas para tener una foto rápida del desempeño


## 7. Modelo 2 — Árbol de Decisión

Este modelo no asume una fórmula lineal: va dividiendo los datos en grupos
haciendo preguntas tipo "¿area > 5000?" hasta llegar a un precio estimado.
Puede capturar relaciones más complejas, pero también se puede "memorizar"
los datos de entrenamiento (overfitting) si lo dejo crecer sin control.

In [ ]:
modelo_arbol = DecisionTreeRegressor(
    max_depth=6,       # limito qué tan profundo puede crecer el árbol; si no le pongo límite, se aprende de memoria los datos de entrenamiento y luego falla con datos nuevos
    random_state=42    # misma idea que antes: fijo la semilla para que el resultado sea reproducible
)

modelo_arbol.fit(X_entrenamiento, y_entrenamiento)  # entreno el árbol con los mismos datos de entrenamiento que usé en la regresión lineal, para que la comparación sea justa

predicciones_arbol = modelo_arbol.predict(X_prueba)  # genero las predicciones sobre el mismo conjunto de prueba
predicciones_arbol[:5]  # miro las primeras 5 para comparar "a ojo" contra las de la regresión lineal más adelante


In [ ]:
mae_arbol = mean_absolute_error(y_prueba, predicciones_arbol)  # mismo cálculo que antes pero con las predicciones del árbol
rmse_arbol = np.sqrt(mean_squared_error(y_prueba, predicciones_arbol))  # RMSE del árbol
r2_arbol = r2_score(y_prueba, predicciones_arbol)  # R2 del árbol

print(f"Árbol de Decisión -> MAE: {mae_arbol:,.0f} | RMSE: {rmse_arbol:,.0f} | R2: {r2_arbol:.3f}")  # imprimo igual que con el modelo anterior para poder comparar los números directamente


## 8. Comparar el rendimiento de los dos modelos

Ahora pongo las métricas una al lado de la otra para decidir, con números y
no solo con intuición, cuál modelo se comporta mejor en este dataset.

In [ ]:
resultados = pd.DataFrame({
    "Modelo": ["Regresión Lineal", "Árbol de Decisión"],  # nombro cada fila para que la tabla se entienda sola
    "MAE": [mae_lineal, mae_arbol],        # junto el error absoluto promedio de los dos modelos
    "RMSE": [rmse_lineal, rmse_arbol],     # junto el RMSE de los dos modelos
    "R2": [r2_lineal, r2_arbol]            # junto el R2 de los dos modelos
})

resultados  # muestro la tabla final de comparación, esta es la que realmente responde la pregunta del ejercicio


In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(11, 4))  # creo una figura con dos gráficas lado a lado: una para RMSE y otra para R2

ejes[0].bar(resultados["Modelo"], resultados["RMSE"], color=["#4C72B0", "#DD8452"])  # barra comparando el error (RMSE): mientras más bajo, mejor
ejes[0].set_title("RMSE por modelo (menor es mejor)")  # título explicando cómo leer esta gráfica

ejes[1].bar(resultados["Modelo"], resultados["R2"], color=["#4C72B0", "#DD8452"])  # barra comparando el R2: mientras más cercano a 1, mejor
ejes[1].set_title("R2 por modelo (mayor es mejor)")  # título explicando la segunda gráfica

plt.tight_layout()  # ajusto los espacios para que los títulos no se encimen entre las dos gráficas
plt.show()  # muestro las dos gráficas juntas


In [ ]:
importancia_variables = pd.Series(
    modelo_arbol.feature_importances_,  # el árbol guarda internamente qué tanto usó cada variable para dividir los datos
    index=caracteristicas.columns        # le pongo el nombre real de cada variable en vez de un índice numérico
).sort_values(ascending=False)  # ordeno de la variable más importante a la menos importante, para leerlo de arriba hacia abajo

importancia_variables  # muestro cuáles variables el árbol consideró más decisivas para el precio


## 9. Análisis y conclusiones (a nivel personal)

Esto es lo que yo entiendo después de correr este notebook, en mis palabras:

- **¿Cuál modelo tuvo mejor rendimiento?** Hay que mirar la tabla de la sección 8:
  el modelo con **menor RMSE/MAE** y **mayor R²** es, en números, el que mejor
  generaliza sobre casas que no vio durante el entrenamiento. En mis corridas,
  la Regresión Lineal y el Árbol quedaron relativamente cerca, pero el que gane
  puede cambiar un poco según la semilla (`random_state`) o el `max_depth` que
  use en el árbol — por eso no me quedo con un solo número, sino que reviso
  las tres métricas juntas.

- **¿Por qué podría ganar la Regresión Lineal?** Porque este dataset es
  relativamente pequeño (545 filas) y las relaciones entre área/precio parecen
  bastante lineales (lo vi en el heatmap de correlación). Cuando la relación
  real es casi una línea recta, un modelo lineal simple puede generalizar mejor
  que un árbol, que tiende a "memorizar" ruido si no se controla su profundidad.

- **¿Por qué podría ganar el Árbol de Decisión?** Porque puede capturar
  relaciones que no son una línea recta (por ejemplo, que tener aire
  acondicionado valga más solo si la casa además es grande). Si le subo el
  `max_depth`, probablemente baje el error en el set de entrenamiento, pero
  corro el riesgo de overfitting: que le vaya peor en el set de prueba.

- **Variables más importantes:** según la sección 8, el árbol confirma lo que
  ya sospechaba con la correlación: `area` suele ser la variable que más pesa,
  seguida de baños, aire acondicionado o el estado del amoblado, dependiendo
  de la corrida.

- **Qué probaría después:** compararía estos dos modelos contra un
  `RandomForestRegressor` (que combina muchos árboles y normalmente reduce el
  overfitting de un árbol solo), y también probaría con `GridSearchCV` para
  buscar automáticamente el mejor `max_depth` en vez de dejarlo fijo en 6.

**Conclusión final:** para este dataset específico, ningún modelo es
"siempre mejor" — la Regresión Lineal me da un resultado más estable y fácil
de interpretar (sé exactamente cuánto pesa cada variable), mientras que el
Árbol de Decisión es más flexible pero necesito cuidar su profundidad para
que no memorice de más. La tabla de la sección 8 es la que manda la última
palabra según los números de esta corrida en particular.
